In [1]:
# To run only once or else restart the kernel
# or change manually the current directory so that it is Networks_project/
import os
os.chdir("../")

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from src.geography_utils import import_location_data, plot_type_repartion_log

## Importation de la donnée

On regroupe par lieu et on compte le nombre d'utilisateurs uniques qui sont venus.

In [3]:
path_data_check_in='data/dataset_TSMC2014_TKY.txt'

df = import_location_data(path_data_check_in)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 67805 entries, 0 to 67804
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   location_id         67805 non-null  str    
 1   latitude            67805 non-null  float64
 2   longitude           67805 non-null  float64
 3   location_type_ID    67805 non-null  str    
 4   location_type_name  67805 non-null  str    
 5   nb_checkins         67805 non-null  int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 3.1 MB


## Premier parcours de la donnée

On ne conserve que les types de lieux très visités pour la répartition par type :

In [5]:
df_loc_type = df.groupby("location_type_ID")["nb_checkins"].agg("sum")
threshold = df_loc_type.sum()/100
filtered_types = list(df_loc_type[df_loc_type >= threshold].index)
print(f"Number of location types with number of check-ins above the threshold ({threshold}): {len(filtered_types)}.")

Number of location types with number of check-ins above the threshold (2199.34): 23.


In [6]:
fig = plot_type_repartion_log(df, filtered_types)

fig.show()

Les services de restauration sont très présents à faible nombre de check-ins, et les gares à fort nombre de check-ins.

In [7]:
fig = px.scatter_map(
    df[df["nb_checkins"] >= 100],
    lat="latitude",
    lon="longitude",
    title="Locations in Tokyo with more than 100 checkins",
    color="location_type_name",
    size="nb_checkins",
    size_max=25,
    zoom=10,
    map_style="carto-positron"
)
fig.show()

Il est clair: ce sont les gares qui sont les plus visitées. Etant donné que celles-ci sont très fréquentées au quotidien et, qu'au Japon, elles proposent souvent des tampons à collectionner dans des carnets, il est courant de partager ses collectes de tampons sur les réseaux. Cela pourrait donc expliquer le grand nombre de check-ins pour ce type de lieu. 

## Corrélation géographique

On remarque également sur la carte précédente qu'il y a une forte concentration de lieux attractifs dans certains quartiers (Shibuya, Shinjuku, ...) : il est donc pertinant d'étudier la corrélation géographique.

In [63]:
import geopandas as gpd
from shapely.geometry import Point

# Conversion rapide : création de la colonne geometry
gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326" # Coordonnées GPS standards
)

# Si vous voulez des distances précises en mètres (ex: France)
# gdf = gdf.to_crs(epsg=2154)

In [ ]:
import pandas as pd
import numpy as np
from libpysal.weights import KNN, DistanceBand, Kernel
from esda.moran import Moran

# 1. Préparation des données
# On utilise log1p à cause de la Power Law (distribution de Pareto)
y = np.log1p(df['nb_checkins']).values
coords = df[['longitude', 'latitude']].values

results = {}

# --- MÉTHODE 1 : K-Nearest Neighbors (KNN) ---
# Idéal pour Tokyo car le nombre de voisins est constant malgré les variations de densité.
# On prend k=8 pour capturer l'entourage immédiat en milieu urbain.
w_knn = KNN.from_array(coords, k=8)
w_knn.transform = 'R'
results['KNN (k=8)'] = Moran(y, w_knn)

# --- MÉTHODE 2 : Distance Band (Rayon fixe) ---
# Utile pour voir l'influence physique réelle. 
# Attention : à Tokyo, 0.01 degré ~ 1.1km. 
# On choisit 500m (env. 0.0045) pour l'influence de quartier.
w_dist = DistanceBand.from_array(coords, threshold=0.0045)
w_dist.transform = 'R'
results['Distance Band (500m)'] = Moran(y, w_dist)

# --- MÉTHODE 3 : Kernel (Poids dégressif) ---
# Plus on est loin, moins on influence. C'est le plus réaliste pour 
# modéliser l'attractivité d'un quartier commerçant.
w_kernel = Kernel.from_array(coords, k=12, function='triangular')
w_kernel.transform = 'R'
results['Kernel (Triangulaire)'] = Moran(y, w_kernel)

# 2. Affichage des résultats
print(f"{'Méthode':<25} | {'Indice I':<10} | {'p-value':<10}")
print("-" * 50)
for name, m in results.items():
    print(f"{name:<25} | {m.I:>10.4f} | {m.p_sim:>10.4f}")

/home/onyxia/work/Networks_projet/.venv/lib/python3.13/site-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)
/home/onyxia/work/Networks_projet/.venv/lib/python3.13/site-packages/libpysal/weights/util.py:819: UserWarning: The weights matrix is not fully connected: 
 There are 250 disconnected components.
 There are 117 islands with ids: 1921, 2676, 3154, 3194, 8442, 12895, 14138, 15061, 15100, 15552, 15618, 16738, 17006, 17154, 19640, 21035, 22036, 22585, 23432, 23672, 24760, 25058, 25066, 25173, 25260, 25718, 26030, 26124, 26315, 27415, 27449, 27651, 28356, 28825, 29531, 30000, 31694, 32549, 32585, 32977, 33746, 33857, 34252, 35607, 35612, 36017, 36329, 36510, 36538, 36826, 36838, 37425, 37541, 37544, 37590, 39595, 40811, 40825, 41096, 41505, 41981, 43232, 43848, 44678, 44753, 45297, 45466, 46083, 46119, 46226, 47008, 49432, 49981, 50054, 50246,

('WARNING: ', 1921, ' is an island (no neighbors)')
('WARNING: ', 2676, ' is an island (no neighbors)')
('WARNING: ', 3154, ' is an island (no neighbors)')
('WARNING: ', 3194, ' is an island (no neighbors)')
('WARNING: ', 8442, ' is an island (no neighbors)')
('WARNING: ', 12895, ' is an island (no neighbors)')
('WARNING: ', 14138, ' is an island (no neighbors)')
('WARNING: ', 15061, ' is an island (no neighbors)')
('WARNING: ', 15100, ' is an island (no neighbors)')
('WARNING: ', 15552, ' is an island (no neighbors)')
('WARNING: ', 15618, ' is an island (no neighbors)')
('WARNING: ', 16738, ' is an island (no neighbors)')
('WARNING: ', 17006, ' is an island (no neighbors)')
('WARNING: ', 17154, ' is an island (no neighbors)')
('WARNING: ', 19640, ' is an island (no neighbors)')
('WARNING: ', 21035, ' is an island (no neighbors)')
('WARNING: ', 22036, ' is an island (no neighbors)')
('WARNING: ', 22585, ' is an island (no neighbors)')
('WARNING: ', 23432, ' is an island (no neighbors)'

## Clustering géographique

On remarque également sur la carte précédente qu'il y a une forte concentration de lieux attractifs dans certains quartiers (Shibuya, Shinjuku, ...) : il est donc pertinant de faire du clustering géographique

La particularité du clustering géographique est qu'il doit assurer une cohérence spatiale tout en prenant en compte les similarités au niveau de la valeur cible (ici, le nombre de visiteurs). Il se pose également la question de la gestion des outliers. Un algorithme de clustering très connu et adapté à cette situation est alors HDBSCAN : utilisons-le.

In [52]:
from sklearn.cluster import HDBSCAN

clustering = HDBSCAN(
    min_cluster_size=200,
    metric='euclidean'
)

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

df["log_checkins"] = np.log(1 + df["nb_checkins"])
X = df[["latitude", "longitude", "log_checkins"]].values
X = StandardScaler().fit_transform(X)

In [53]:
y = clustering.fit_predict(X)

/home/onyxia/work/Networks_projet/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


In [54]:
df["cluster"] = y

In [55]:
df["cluster"].nunique()

39

In [56]:
import numpy as np

# 2. Filtrer pour exclure les points sans cluster (-1)
df_clusters_only = df[df['cluster'] != -1]

# 3. Agrégation : Calcul des centres et du nombre de points
df_agg = df_clusters_only.groupby('cluster').agg(
    center_lat=('latitude', 'mean'),
    center_lon=('longitude', 'mean'),
    checkins_count=('nb_checkins', 'sum')
).reset_index()

# 4. Calcul du rayon (distance maximale au centre)
def calculate_radius(group):
    # On récupère le centre du cluster correspondant dans df_agg
    c_lat = df_agg.loc[df_agg['cluster'] == group.name, 'center_lat'].values[0]
    c_lon = df_agg.loc[df_agg['cluster'] == group.name, 'center_lon'].values[0]
    
    # Calcul de la distance euclidienne simplifiée (ou utilisez Haversine pour la précision)
    distances = np.sqrt((group['latitude'] - c_lat)**2 + (group['longitude'] - c_lon)**2)
    return distances.max()

# On ajoute le rayon au dataset agrégé
radii = df_clusters_only.groupby('cluster').apply(calculate_radius)
df_agg['radius'] = radii.values

In [57]:
df_agg.columns

Index(['cluster', 'center_lat', 'center_lon', 'checkins_count', 'radius'], dtype='str')

In [58]:
import plotly.express as px

fig = px.scatter_mapbox(
    df_agg, 
    lat="center_lat", 
    lon="center_lon", 
    size="radius",          # La taille du cercle dépend de l'étendue du cluster
    color="checkins_count",     # Une couleur par cluster
    hover_data={'cluster': False, 'center_lat': False, 'center_lon': False, 'checkins_count': True, 'radius': True},
    zoom=10, 
    mapbox_style="carto-positron"
)

fig.show()

/tmp/ipykernel_47862/3365276889.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
